# AlphaBetter 蛋白质稳定性分析 — 演示 Notebook

本 Notebook 演示 AlphaBetter 的完整工作流程：
1. 读取 PDB 文件
2. 识别二级结构（H/E/C）
3. 用二肽/五肽参考数据库打分
4. 输出残基级稳定性评分和彩色 PDB

In [ ]:
import sys
from pathlib import Path

# 把项目根目录加入 Python 路径
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print('项目根目录:', PROJECT_ROOT)

## 1. 运行稳定性计算

In [ ]:
from calc_stability import calc_stability

pdb_file = PROJECT_ROOT / 'test.pdb'
di_db = PROJECT_ROOT / 'data' / 'db_dipeptides.csv'
penta_db = PROJECT_ROOT / 'data' / 'db_pentapeptides.csv'
out_dir = PROJECT_ROOT / 'results'

calc_stability(
    pdb_file=str(pdb_file),
    di_csv=str(di_db),
    penta_csv=str(penta_db),
    out_dir=str(out_dir),
)
print('计算完成！')

## 2. 查看残基评分

In [ ]:
import pandas as pd

df = pd.read_csv(out_dir / 'residue_scores.csv', encoding='utf-8-sig')
print(f'总残基数: {len(df)}')
df.head(10)

In [ ]:
# 统计
positive = (df['score'] > 0.05).sum()
negative = (df['score'] < -0.05).sum()
neutral = len(df) - positive - negative

print(f'稳定区域 (评分 > 0.05): {positive} 个残基')
print(f'不稳定区域 (评分 < -0.05): {negative} 个残基')
print(f'中性区域: {neutral} 个残基')
print(f'评分范围: [{df["score"].min():.3f}, {df["score"].max():.3f}]')

## 3. 查看命中数据库的片段

In [ ]:
hits = pd.read_csv(out_dir / 'hits.csv', encoding='utf-8-sig')
print(f'总命中片段数: {len(hits)}')
hits.head(10)

## 4. 生成彩色 PDB（B-factor 着色）

In [ ]:
from utils import read_residue_scores, write_b_factor

score_map = read_residue_scores(out_dir / 'residue_scores.csv')
colored_pdb = out_dir / 'colored_test.pdb'

matched = write_b_factor(pdb_file, colored_pdb, score_map)
print(f'匹配残基数: {matched}')
print(f'彩色 PDB 已保存: {colored_pdb}')
print('可用 PyMOL/ChimeraX 打开查看 B-factor 着色。')